# LLM Agentic Legal Information Retrieval

This notebook implements an agentic legal citation retrieval system using BM25 + Qwen 3 0.6B for Swiss German law (Kaggle competition). It combines query expansion, direct citation extraction, and optional LLM re-scoring.

In [ ]:
# =========================================================
# LLM AGENTIC LEGAL INFORMATION RETRIEVAL
# Full Improved Kaggle Notebook Code
# BM25 + Qwen 3 0.6B LLM Query Expansion
# Direct Citation Priority + Strong Noise Filtering
# Offline Kaggle Compatible
# =========================================================




## 1. Import Libraries

Standard imports: `os`, `re`, `json`, `math`, numpy, pandas, and utilities. Warnings are suppressed for cleaner output.

In [ ]:
# =========================================================
# 1. IMPORT LIBRARIES
# =========================================================

import os
import re
import gc
import json
import math
import warnings
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

## 2. Load Competition Data

Loads the four CSV files from `/kaggle/input`: training questions, test questions, sample submission, and the German law corpus (`laws_de.csv`).

In [ ]:
DATA_DIR = Path("data")

train_path = DATA_DIR / "train_sample.csv"
test_path = DATA_DIR / "test_sample.csv"
sample_path = DATA_DIR / "sample_submission.csv"
laws_path = DATA_DIR / "laws_sample.csv"

train = pd.read_csv(train_path, header=None, names=["query_id", "query", "gold_citations"])
test = pd.read_csv(test_path, header=None, names=["query_id", "query"])
sample = pd.read_csv(sample_path)
laws = pd.read_csv(laws_path, header=None, names=["citation", "text", "title"])

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Sample shape:", sample.shape)
print("Laws shape:", laws.shape)

print("\nTrain columns:", train.columns.tolist())
print("Test columns:", test.columns.tolist())
print("Sample columns:", sample.columns.tolist())
print("Laws columns:", laws.columns.tolist())

## 3. Column Detection

Dynamically detects the correct column names (e.g. `query` vs `question`, `gold_citations` vs `citations`) so the notebook works across different dataset versions.

In [ ]:
# =========================================================
# 3. COLUMN DETECTION
# =========================================================

def pick_col(df, possible_cols):
    for col in possible_cols:
        if col in df.columns:
            return col
    raise ValueError(f"None of these columns found: {possible_cols}")


question_col = pick_col(train, ["query", "question", "legal_question", "Question"])
gold_col = pick_col(train, ["gold_citations", "citations", "answers", "Gold"])

citation_col = pick_col(laws, ["citation", "Citation", "source", "id"])
text_col = pick_col(laws, ["text", "Text", "content", "law_text"])
title_col = "title" if "title" in laws.columns else text_col

id_col = "query_id" if "query_id" in test.columns else sample.columns[0]
pred_col = "predicted_citations" if "predicted_citations" in sample.columns else sample.columns[1]

print("\nDetected columns:")
print("question_col:", question_col)
print("gold_col:", gold_col)
print("citation_col:", citation_col)
print("text_col:", text_col)
print("title_col:", title_col)
print("id_col:", id_col)
print("pred_col:", pred_col)

## 4. Prepare Law Corpus

Builds the searchable text for each law article by concatenating citation ID + title + article text. Also creates lookup structures (`citation_to_index`, `valid_law_citations`).

In [ ]:
# =========================================================
# 4. PREPARE LAW CORPUS
# =========================================================

laws = laws.dropna(subset=[citation_col, text_col]).reset_index(drop=True)

law_citations = laws[citation_col].astype(str).str.strip().tolist()

law_texts = (
    laws[citation_col].fillna("").astype(str) + " " +
    laws[title_col].fillna("").astype(str) + " " +
    laws[text_col].fillna("").astype(str)
).tolist()

valid_law_citations = set(law_citations)

citation_to_index = {
    citation: i for i, citation in enumerate(law_citations)
}

print("\nNumber of legal documents:", len(law_texts))
print("Example citation:", law_citations[0])
print("Example law text:", law_texts[0][:300])

## 5. Tokenizer

A lightweight regex tokenizer that lowercases text and extracts alphanumeric tokens (including accented characters for German support).

In [ ]:
# =========================================================
# 5. TOKENIZER
# =========================================================

def tokenize(text):
    text = str(text).lower()
    tokens = re.findall(r"[a-zA-ZÀ-ÿ0-9_]+", text)
    return tokens

## 6. BM25 Class

Custom BM25 implementation (k1=1.5, b=0.75). Uses inverted postings for efficient scoring. This is the core retrieval engine for both law documents and training queries.

In [ ]:
# =========================================================
# 6. BM25 CLASS
# =========================================================

class BM25:
    def __init__(self, documents, k1=1.5, b=0.75):
        self.k1 = k1
        self.b = b

        self.docs = [tokenize(doc) for doc in documents]
        self.N = len(self.docs)

        self.doc_len = np.array([len(doc) for doc in self.docs], dtype=np.float32)
        self.avgdl = self.doc_len.mean() if self.N > 0 else 1.0

        self.df = Counter()
        self.postings = defaultdict(list)

        for i, doc in enumerate(self.docs):
            tf = Counter(doc)
            for term, freq in tf.items():
                self.df[term] += 1
                self.postings[term].append((i, freq))

        self.idf = {}

        for term, df in self.df.items():
            self.idf[term] = math.log(1 + (self.N - df + 0.5) / (df + 0.5))

    def get_scores(self, query):
        query_terms = tokenize(query)
        scores = np.zeros(self.N, dtype=np.float32)

        for term in query_terms:
            if term not in self.postings:
                continue

            idf = self.idf.get(term, 0.0)

            for doc_id, freq in self.postings[term]:
                dl = self.doc_len[doc_id]
                numerator = freq * (self.k1 + 1)
                denominator = freq + self.k1 * (1 - self.b + self.b * dl / self.avgdl)
                scores[doc_id] += idf * numerator / denominator

        return scores

## 7. Build BM25 Index

Indexes all law articles using the BM25 class above. This is the main retrieval index against which queries will be scored.

In [ ]:
# =========================================================
# 7. BUILD BM25 INDEX FOR GERMAN LAW CORPUS
# =========================================================

bm25_laws = BM25(law_texts)
print("\nBM25 law index built successfully.")

## 8. English → German Legal Dictionary

A hand-crafted bilingual dictionary mapping English legal concepts to German terms and Swiss law codes (e.g. `'contract'` → `'Vertrag OR'`). Also defines `law_area_keywords` for code-to-concept mapping.

In [ ]:
# =========================================================
# 8. ENGLISH TO GERMAN LEGAL DICTIONARY
# =========================================================

legal_dictionary = {
    # Contract / obligation law
    "contract": "Vertrag Vereinbarung Obligation OR",
    "agreement": "Vertrag Vereinbarung OR",
    "obligation": "Obligation Schuld OR",
    "liability": "Haftung Verantwortlichkeit Schaden OR",
    "damage": "Schaden Ersatz Haftung OR",
    "damages": "Schadenersatz Haftung OR",
    "compensation": "Entschädigung Schadenersatz OR",
    "termination": "Kündigung Beendigung Auflösung OR",
    "unfair termination": "missbräuchliche Kündigung OR",
    "employment": "Arbeitsvertrag Arbeitnehmer Arbeitgeber OR",
    "employee": "Arbeitnehmer OR",
    "employer": "Arbeitgeber OR",
    "salary": "Lohn Arbeitsvertrag Arbeitnehmer OR",
    "wage": "Lohn Arbeitsvertrag Arbeitnehmer OR",
    "dismissal": "Kündigung Entlassung Arbeitsvertrag OR",
    "lease": "Miete Mietvertrag OR",
    "rent": "Miete Mietzins OR",
    "sale": "Kauf Kaufvertrag OR",
    "purchase": "Kauf Kaufvertrag OR",
    "company": "Gesellschaft Unternehmen OR",
    "corporation": "Aktiengesellschaft Gesellschaft OR",
    "shareholder": "Aktionär Gesellschaft OR",
    "agency": "Auftrag Stellvertretung OR",
    "mandate": "Auftrag OR",
    "loan": "Darlehen OR",
    "guarantee": "Bürgschaft Garantie OR",
    "remission of debt": "Schulderlass Erlassvertrag OR",
    "novation": "Novation Schuldner Schuldanerkennung OR ZGB",
    "consumer credit": "Konsumkredit KKG",

    # Civil code
    "marriage": "Ehe Ehegatte ZGB",
    "divorce": "Scheidung Ehe ZGB",
    "child": "Kind Eltern ZGB",
    "parent": "Eltern Kind ZGB",
    "custody": "elterliche Sorge Obhut Sorgerecht ZGB",
    "inheritance": "Erbschaft Erbe Erbrecht ZGB",
    "estate": "Nachlass Erbschaft ZGB",
    "will": "Testament letztwillige Verfügung ZGB",
    "property": "Eigentum Grundstück Besitz ZGB",
    "ownership": "Eigentum Besitz ZGB",
    "land register": "Grundbuch ZGB",
    "mortgage": "Grundpfand Schuldbrief Hypothek ZGB",
    "mortgage certificate": "Schuldbrief Inhaberschuldbrief Grundpfand ZGB SchKG",
    "personality": "Persönlichkeit Persönlichkeitsschutz ZGB",
    "name": "Name Namensrecht ZGB",
    "association": "Verein ZGB",
    "foundation": "Stiftung ZGB",

    # Criminal law
    "criminal": "strafbar Strafe Strafgesetzbuch StGB",
    "crime": "Straftat strafbar StGB",
    "offence": "Straftat strafbar StGB",
    "offense": "Straftat strafbar StGB",
    "murder": "Tötung Mord StGB",
    "homicide": "Tötung StGB",
    "theft": "Diebstahl StGB",
    "fraud": "Betrug StGB",
    "assault": "Körperverletzung StGB",
    "bodily harm": "Körperverletzung StGB",
    "sexual": "sexuelle Handlung Sexualdelikt StGB",
    "defamation": "Ehrverletzung Verleumdung üble Nachrede StGB",
    "forgery": "Urkundenfälschung StGB",
    "money laundering": "Geldwäscherei StGB",
    "trade secret": "Geschäftsgeheimnis Fabrikationsgeheimnis UWG StGB",

    # Courts / procedure
    "appeal": "Beschwerde Berufung Rekurs BGG",
    "federal court": "Bundesgericht BGG",
    "federal tribunal": "Bundesgericht BGG",
    "court": "Gericht Bundesgericht Verfahren",
    "decision": "Entscheid Urteil",
    "judgment": "Urteil Entscheid",
    "admissible": "zulässig Zulässigkeit BGG",
    "procedure": "Verfahren Prozess ZPO StPO VwVG",
    "civil procedure": "Zivilprozess ZPO",
    "criminal procedure": "Strafprozess Strafprozessordnung StPO",
    "administrative procedure": "Verwaltungsverfahren VwVG",
    "injunction": "vorsorgliche Massnahmen Verbot Unterlassung ZPO",
    "interim relief": "vorsorgliche Massnahmen ZPO",
    "provisional measures": "vorsorgliche Massnahmen ZPO",
    "seizure": "Beschlagnahme Sicherstellung Beweissicherung ZPO",
    "forensic inspection": "Beweissicherung Edition Augenschein ZPO",
    "competent authority": "Zuständigkeit Gericht Behörde IPRG ZPO",
    "standing": "Aktivlegitimation Klageberechtigung Rechtsinhaber",

    # Constitution
    "constitutional": "Verfassung Bundesverfassung BV",
    "municipality": "Gemeinde BV",
    "canton": "Kanton BV",
    "fundamental right": "Grundrecht BV",
    "freedom": "Freiheit Grundrecht BV",
    "equality": "Gleichheit Rechtsgleichheit BV",
    "religion": "Religionsfreiheit BV",
    "expression": "Meinungsfreiheit BV",
    "privacy": "Privatsphäre Datenschutz BV",

    # Environment
    "environment": "Umwelt Umweltschutz USG",
    "environmental": "Umwelt Umweltschutz USG",
    "impact assessment": "Umweltverträglichkeitsprüfung UVPV",
    "pollution": "Verschmutzung Emission Immission USG",
    "noise": "Lärm Lärmschutz USG",
    "waste": "Abfall Entsorgung USG",
    "water": "Gewässer Wasser GSchG",
    "forest": "Wald WaG",

    # Insurance / road traffic / social security
    "insurance": "Versicherung UVG KVG AHVG IVG",
    "accident": "Unfall Unfallversicherung UVG SVG",
    "traffic accident": "Verkehrsunfall Motorfahrzeug Halter SVG",
    "vehicle holder": "Halter Motorfahrzeug SVG",
    "gross negligence": "grobe Fahrlässigkeit SVG",
    "health insurance": "Krankenversicherung KVG",
    "disability": "Invalidität Invalidenversicherung IVG",
    "pension": "Altersversicherung AHV berufliche Vorsorge BVG",
    "unemployment": "Arbeitslosenversicherung AVIG",

    # International private law
    "international": "international Ausland IPRG",
    "foreign": "Ausland international IPRG",
    "arbitration": "Schiedsgericht Schiedsverfahren IPRG",
    "arbitral": "Schiedsgericht Schiedsverfahren IPRG",
    "jurisdiction": "Zuständigkeit Gerichtsstand IPRG LugÜ",
    "recognition": "Anerkennung Vollstreckung IPRG",

    # Immigration / asylum
    "immigration": "Ausländer Aufenthalt Bewilligung AIG",
    "foreigner": "Ausländer Aufenthalt Bewilligung AIG",
    "residence permit": "Aufenthaltsbewilligung Niederlassungsbewilligung AIG",
    "asylum": "Asyl Flüchtling AsylG",

    # Tax
    "tax": "Steuer Besteuerung DBG StHG MWSTG",
    "income tax": "Einkommenssteuer DBG",
    "vat": "Mehrwertsteuer MWSTG",

    # Data / IP / competition
    "data protection": "Datenschutz DSG",
    "personal data": "Personendaten Datenschutz DSG",
    "copyright": "Urheberrecht URG Werk Computerprogramm Quellcode",
    "source code": "Quellcode Computerprogramm Software URG",
    "software": "Software Computerprogramm Quellcode URG",
    "rights-holder": "Rechtsinhaber Urheber Inhaber URG",
    "trademark": "Marke Markenschutz MSchG",
    "patent": "Patent Patentrecht PatG",
    "confidential": "vertraulich Geschäftsgeheimnis Geheimnis UWG",
    "unfair competition": "unlauterer Wettbewerb UWG",

    # Bankruptcy / debt enforcement
    "bankruptcy": "Konkurs SchKG",
    "opposition": "Rechtsvorschlag Rechtsöffnung SchKG",
    "payment order": "Zahlungsbefehl Betreibung SchKG",
    "debt enforcement": "Betreibung Schuldbetreibung SchKG",
    "solvency": "Zahlungsfähigkeit Zahlungsunfähigkeit Konkurs SchKG",
    "time-barred": "verjährt Verjährung Frist OR SVG",
    "prescription": "Verjährung Verwirkungsfrist Frist OR SVG",
}


law_area_keywords = {
    "OR": [
        "contract", "agreement", "obligation", "liability", "damage", "damages",
        "compensation", "termination", "employment", "employee", "employer",
        "salary", "wage", "dismissal", "lease", "rent", "sale", "purchase",
        "company", "corporation", "shareholder", "agency", "mandate", "loan",
        "guarantee", "remission of debt", "novation"
    ],
    "ZGB": [
        "marriage", "divorce", "child", "parent", "custody", "inheritance",
        "estate", "will", "property", "ownership", "land register", "mortgage",
        "mortgage certificate", "personality", "name", "association", "foundation"
    ],
    "StGB": [
        "criminal", "crime", "offence", "offense", "murder", "homicide",
        "theft", "fraud", "assault", "bodily harm", "sexual", "defamation",
        "forgery", "money laundering", "trade secret"
    ],
    "BGG": ["appeal", "federal court", "federal tribunal", "admissible", "admissibility"],
    "BV": ["constitutional", "municipality", "canton", "fundamental right", "freedom", "equality", "religion", "expression", "privacy"],
    "ZPO": ["injunction", "interim relief", "provisional measures", "seizure", "forensic inspection", "civil procedure"],
    "USG": ["environment", "environmental", "pollution", "noise", "waste"],
    "UVPV": ["impact assessment", "environmental impact"],
    "GSchG": ["water"],
    "WaG": ["forest"],
    "UVG": ["insurance", "accident"],
    "SVG": ["traffic accident", "vehicle holder", "gross negligence", "accident"],
    "KVG": ["health insurance", "insurance"],
    "AHVG": ["pension", "social security"],
    "IVG": ["disability"],
    "BVG": ["pension"],
    "AVIG": ["unemployment"],
    "IPRG": ["international", "foreign", "arbitration", "arbitral", "jurisdiction", "recognition", "competent authority"],
    "AIG": ["immigration", "foreigner", "residence permit"],
    "AsylG": ["asylum"],
    "DBG": ["tax", "income tax"],
    "StHG": ["tax"],
    "MWSTG": ["vat", "tax"],
    "DSG": ["data protection", "personal data"],
    "URG": ["copyright", "source code", "software", "rights-holder"],
    "UWG": ["unfair competition", "trade secret", "confidential"],
    "MSchG": ["trademark"],
    "PatG": ["patent"],
    "SchKG": ["bankruptcy", "opposition", "payment order", "debt enforcement", "solvency"],
    "KKG": ["consumer credit"],
}

## 9. Law Code Extraction

Extracts law codes (e.g. `OR`, `ZGB`, `StGB`) from citation strings. Handles aliases (e.g. `CO` → `OR`) and builds a per-code inverted index for boosting.

In [ ]:
# =========================================================
# 9. LAW CODE EXTRACTION
# =========================================================

roman_numerals = {"I", "II", "III", "IV", "V", "VI", "VII", "VIII", "IX", "X"}

law_code_alias = {
    "CO": "OR",
    "LCC": "KKG",
    "CC": "ZGB",
    "SCC": "ZGB",
    "CP": "StGB",
}


def normalize_law_code(code):
    code = str(code).strip()
    return law_code_alias.get(code, code)


def extract_law_code_from_citation(citation):
    citation = str(citation).strip()
    parts = citation.split()

    if len(parts) == 0:
        return None

    last = parts[-1].strip()

    if last in roman_numerals:
        return None

    last = normalize_law_code(last)

    if re.fullmatch(r"[A-ZÄÖÜ][A-Za-zÄÖÜäöü]{1,12}G?", last):
        return last

    return None


detected_codes = []

for c in law_citations:
    code = extract_law_code_from_citation(c)
    if code is not None:
        detected_codes.append(code)

known_law_codes = sorted(set(list(law_area_keywords.keys()) + detected_codes + list(law_code_alias.keys())))

print("\nNumber of known law codes:", len(known_law_codes))
print("Some law codes:", known_law_codes[:50])


law_code_indices = {}

for code in known_law_codes:
    normalized = normalize_law_code(code)
    pattern = rf"\b{re.escape(normalized)}\b"

    indices = [
        i for i, citation in enumerate(law_citations)
        if re.search(pattern, str(citation))
    ]

    law_code_indices[normalized] = np.array(indices, dtype=int)

print("Law-code index built.")

## 10. Dictionary-Based Query Expansion

Expands English queries with German legal terms using the dictionary above. Also detects which Swiss law codes are likely relevant based on keywords in the query.

In [ ]:
# =========================================================
# 10. QUERY EXPANSION WITHOUT LLM
# =========================================================

def detect_target_laws(query):
    query_lower = str(query).lower()
    target_laws = []

    for law_code, keywords in law_area_keywords.items():
        for keyword in keywords:
            if keyword in query_lower:
                target_laws.append(law_code)
                break

    query_upper = str(query).upper()

    for code in known_law_codes:
        if re.search(rf"\b{re.escape(code.upper())}\b", query_upper):
            target_laws.append(normalize_law_code(code))

    return list(set(target_laws))


def expand_query_dictionary(query):
    query_lower = str(query).lower()
    extra_terms = []

    for eng, ger in legal_dictionary.items():
        if eng in query_lower:
            extra_terms.append(ger)

    target_laws = detect_target_laws(query)
    extra_terms.extend(target_laws)

    expanded_query = str(query) + " " + " ".join(extra_terms)

    return expanded_query, target_laws

## 11. Direct Citation Extraction

Parses article references directly mentioned in the query (e.g. `Art. 83 SVG`, `BGE 123 III 456`, case numbers like `4A_100/2020`). These receive a strong boosting score in retrieval.

In [ ]:
# =========================================================
# 11. IMPROVED DIRECT CITATION EXTRACTION
# =========================================================

known_law_codes_regex = sorted(set(known_law_codes + list(law_code_alias.keys())), key=len, reverse=True)

article_index = defaultdict(list)

article_citation_pattern = re.compile(
    r"^Art\.?\s*(?P<article>\d+[a-zA-Z]?)"
    r"(?:\s*Abs\.?\s*(?P<abs>\d+))?"
    r".*?\s(?P<code>[A-Za-zÄÖÜäöü]+)$"
)

for citation in law_citations:
    citation_str = str(citation).strip()
    m = article_citation_pattern.search(citation_str)

    if m:
        article = m.group("article")
        abs_no = m.group("abs")
        code = normalize_law_code(m.group("code"))

        article_index[(article, code)].append(citation_str)

        if abs_no is not None:
            article_index[(article, code, abs_no)].append(citation_str)

print("Article citation index built.")


def find_article_variants(article_number, law_code, abs_no=None, max_variants=6):
    article_number = str(article_number).strip()
    law_code = normalize_law_code(law_code)

    if abs_no is not None:
        variants = article_index.get((article_number, law_code, str(abs_no)), [])
    else:
        variants = article_index.get((article_number, law_code), [])

    variants = sorted(
        list(dict.fromkeys(variants)),
        key=lambda x: (len(x), x)
    )

    return variants[:max_variants]


def extract_direct_citations_from_query(query):
    query = str(query)
    citations = []

    law_code_pattern = "|".join(re.escape(code) for code in known_law_codes_regex)

    article_pattern = re.compile(
        rf"Art\.?\s*(?P<article>\d+[a-zA-Z]?)"
        rf"(?:\s*Abs\.?\s*(?P<abs>\d+))?"
        rf"\s+(?P<code>{law_code_pattern})\b",
        flags=re.IGNORECASE
    )

    for m in article_pattern.finditer(query):
        article = m.group("article")
        abs_no = m.group("abs")
        code = normalize_law_code(m.group("code"))

        possible_exact = []

        if abs_no is not None:
            possible_exact.append(f"Art. {article} Abs. {abs_no} {code}")

        possible_exact.append(f"Art. {article} {code}")

        found = False

        for exact in possible_exact:
            exact = re.sub(r"\s+", " ", exact).strip()

            if exact in valid_law_citations:
                citations.append(exact)
                found = True

        if not found:
            variants = find_article_variants(
                article_number=article,
                law_code=code,
                abs_no=abs_no,
                max_variants=6
            )

            if len(variants) == 0:
                variants = find_article_variants(
                    article_number=article,
                    law_code=code,
                    abs_no=None,
                    max_variants=6
                )

            citations.extend(variants)

    bge_pattern = r"\bBGE\s+\d+\s+[IVX]+\s+\d+\b"

    for m in re.finditer(bge_pattern, query):
        citation = re.sub(r"\s+", " ", m.group(0)).strip()

        if citation in valid_law_citations:
            citations.append(citation)

    case_pattern = r"\b\d+[A-Z]_[0-9]+/[0-9]{4}\b"

    for m in re.finditer(case_pattern, query):
        citation = m.group(0).strip()

        if citation in valid_law_citations:
            citations.append(citation)

    return list(dict.fromkeys(citations))

## 12. Load Offline LLM (Qwen 3 0.6B)

Loads the Qwen 3 0.6B model from the local Kaggle input directory. Falls back gracefully if the model isn't found — but submission requires a successful load.

In [ ]:
# =========================================================
# 12. LOAD OFFLINE QWEN 3 0.6B LLM
# =========================================================

USE_LLM = False
tokenizer = None
llm_model = None

LLM_DIR = Path("/kaggle/input/models/qwen-lm/qwen-3/transformers/0.6b/1")

print("\nLLM_DIR:", LLM_DIR)
print("LLM_DIR exists:", LLM_DIR.exists())

try:
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM

    if not LLM_DIR.exists():
        raise FileNotFoundError(f"LLM folder not found: {LLM_DIR}")

    print("Loading Qwen 3 0.6B from local Kaggle input...")

    tokenizer = AutoTokenizer.from_pretrained(
        LLM_DIR,
        local_files_only=True,
        trust_remote_code=True
    )

    dtype = torch.float16 if torch.cuda.is_available() else torch.float32

    try:
        llm_model = AutoModelForCausalLM.from_pretrained(
            LLM_DIR,
            torch_dtype=dtype,
            device_map="auto",
            local_files_only=True,
            trust_remote_code=True
        )
    except Exception:
        device = "cuda" if torch.cuda.is_available() else "cpu"

        llm_model = AutoModelForCausalLM.from_pretrained(
            LLM_DIR,
            torch_dtype=dtype,
            local_files_only=True,
            trust_remote_code=True
        )

        llm_model = llm_model.to(device)

    llm_model.eval()
    USE_LLM = True

    print("Offline LLM loaded successfully.")
    print("USE_LLM:", USE_LLM)
    print("Model device:", next(llm_model.parameters()).device)

except Exception as e:
    print("LLM loading failed.")
    print("Error:", e)
    print("LLM not available. Running in dictionary-only mode.")
    USE_LLM = False

## 13. LLM Helper Functions

Utilities for: building chat-template prompts, running inference with `torch.no_grad()`, and extracting JSON from raw LLM output.

In [ ]:
# =========================================================
# 13. LLM HELPER FUNCTIONS
# =========================================================

def get_pad_token_id():
    if tokenizer is None:
        return None

    if tokenizer.pad_token_id is not None:
        return tokenizer.pad_token_id

    if tokenizer.eos_token_id is not None:
        return tokenizer.eos_token_id

    return None


def build_prompt(system_prompt, user_prompt):
    messages = [
        {"role": "system", "content": system_prompt.strip()},
        {"role": "user", "content": user_prompt.strip()}
    ]

    if tokenizer is not None and hasattr(tokenizer, "apply_chat_template"):
        try:
            return tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
                enable_thinking=False
            )
        except TypeError:
            return tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True
            )

    return (
        "System:\n" + system_prompt.strip() +
        "\n\nUser:\n" + user_prompt.strip() +
        "\n\nAssistant:\n"
    )


def ask_llm(system_prompt, user_prompt, max_new_tokens=100):
    if not USE_LLM:
        return ""

    try:
        prompt = build_prompt(system_prompt, user_prompt)
        device = next(llm_model.parameters()).device

        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=1800
        )

        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = llm_model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=get_pad_token_id()
            )

        generated_ids = outputs[0][inputs["input_ids"].shape[1]:]
        answer = tokenizer.decode(generated_ids, skip_special_tokens=True)

        return answer.strip()

    except Exception as e:
        print("LLM generation error:", e)
        return ""


def extract_json_from_text(text):
    text = str(text)

    start = text.find("{")
    end = text.rfind("}") + 1

    if start == -1 or end <= start:
        return None

    try:
        return json.loads(text[start:end])
    except Exception:
        return None

## 14. LLM Query Expansion

Uses the LLM to supplement dictionary expansion with additional German legal terms and likely law codes. Results are cached per query to avoid redundant inference.

In [ ]:
# =========================================================
# 14. LLM QUERY EXPANSION
# =========================================================

llm_expansion_cache = {}

def llm_expand_query(query):
    query = str(query)

    if query in llm_expansion_cache:
        return llm_expansion_cache[query]

    dictionary_expanded, dictionary_laws = expand_query_dictionary(query)

    system_prompt = """
You are a Swiss legal information retrieval assistant.
Convert English legal questions into German legal search terms.
Do not answer the question.
Do not invent final citations.
"""

    user_prompt = f"""
English legal question:
{query}

Return only valid JSON in this exact format:
{{
  "german_terms": ["term1", "term2", "term3"],
  "law_codes": ["OR", "ZGB"]
}}

Rules:
- Use German legal vocabulary.
- Give broad Swiss law codes if likely.
- Do not give article numbers.
- Do not explain anything.
"""

    raw = ask_llm(
        system_prompt=system_prompt,
        user_prompt=user_prompt,
        max_new_tokens=120
    )

    data = extract_json_from_text(raw)

    if data is None:
        result = dictionary_expanded, dictionary_laws
        llm_expansion_cache[query] = result
        return result

    german_terms = data.get("german_terms", [])
    law_codes = data.get("law_codes", [])

    german_terms = [str(x).strip() for x in german_terms if str(x).strip()]
    law_codes = [normalize_law_code(str(x).strip()) for x in law_codes if str(x).strip()]

    law_codes = [
        code for code in law_codes
        if code in known_law_codes or len(code) <= 12
    ]

    expanded_query = (
        dictionary_expanded + " " +
        query + " " +
        " ".join(german_terms) + " " +
        " ".join(law_codes)
    )

    final_laws = list(set(dictionary_laws + law_codes))

    result = expanded_query, final_laws
    llm_expansion_cache[query] = result

    return result

## 15. LLM Candidate Re-Scoring (Optional)

Asks the LLM to rate how relevant a candidate citation is to the query (0–3 scale). Used optionally in `rerank_top_n` to re-score the top candidates before final selection.

In [ ]:
# =========================================================
# 15. OPTIONAL LLM CANDIDATE SCORING
# =========================================================

llm_score_cache = {}

def llm_score_candidate(query, citation):
    key = (str(query), str(citation))

    if key in llm_score_cache:
        return llm_score_cache[key]

    idx = citation_to_index.get(citation, None)

    if idx is None:
        title = ""
        text = ""
    else:
        title = str(laws.iloc[idx][title_col])
        text = str(laws.iloc[idx][text_col])[:600]

    system_prompt = """
You are a Swiss legal retrieval judge.
Rate whether the candidate citation is relevant to the legal question.
Return only one digit: 0, 1, 2, or 3.
"""

    user_prompt = f"""
Question:
{query}

Candidate citation:
{citation}

Candidate title:
{title}

Candidate text:
{text}

Score:
0 = not relevant
1 = weakly related
2 = probably relevant
3 = highly relevant

Return only one digit.
"""

    raw = ask_llm(
        system_prompt=system_prompt,
        user_prompt=user_prompt,
        max_new_tokens=5
    )

    match = re.search(r"[0-3]", raw)

    if match:
        score = int(match.group(0))
    else:
        score = 0

    llm_score_cache[key] = score
    return score

## 16. Train Query BM25 + Support Functions

Builds a BM25 index over training questions. Similar training queries are retrieved to transfer their gold citations as candidate suggestions and law code predictions.

In [ ]:
# =========================================================
# 16. TRAIN QUERY BM25
# =========================================================

def parse_citations(x):
    if pd.isna(x):
        return set()

    return set(c.strip() for c in str(x).split(";") if c.strip())


def f1_score_citations(gold, pred):
    gold = set(gold)
    pred = set(pred)

    if len(gold) == 0 and len(pred) == 0:
        return 1.0

    if len(gold) == 0 or len(pred) == 0:
        return 0.0

    tp = len(gold & pred)

    precision = tp / len(pred) if len(pred) > 0 else 0
    recall = tp / len(gold) if len(gold) > 0 else 0

    if precision + recall == 0:
        return 0.0

    return 2 * precision * recall / (precision + recall)


def get_law_codes_from_citations(citation_string):
    citations = parse_citations(citation_string)
    codes = []

    for citation in citations:
        code = extract_law_code_from_citation(citation)
        if code is not None:
            codes.append(code)

    return list(set(codes))


train_queries_full = train[question_col].astype(str).tolist()
train_gold_full = train[gold_col].astype(str).tolist()

query_bm25_full = BM25(train_queries_full)

print("\nTrain-query BM25 index built successfully.")


def retrieve_from_similar_train_queries(query, query_bm25, train_gold_list, top_n_train=5, max_citations=30):
    scores = query_bm25.get_scores(query)

    if len(scores) == 0:
        return [], {}

    top_n_train = min(top_n_train, len(scores))
    top_idx = np.argpartition(scores, -top_n_train)[-top_n_train:]
    top_idx = top_idx[np.argsort(scores[top_idx])[::-1]]

    selected = []
    seen = set()
    support_map = {}

    for rank, idx in enumerate(top_idx):
        golds = parse_citations(train_gold_list[idx])
        weight = 1.0 / (rank + 1)

        for citation in golds:
            support_map[citation] = support_map.get(citation, 0.0) + weight

            if citation not in seen:
                selected.append(citation)
                seen.add(citation)

            if len(selected) >= max_citations:
                break

    return selected, support_map


def predict_law_codes_from_similar_queries(query, query_bm25, train_gold_list, top_n_train=5):
    scores = query_bm25.get_scores(query)

    if len(scores) == 0:
        return []

    top_n_train = min(top_n_train, len(scores))
    top_idx = np.argpartition(scores, -top_n_train)[-top_n_train:]
    top_idx = top_idx[np.argsort(scores[top_idx])[::-1]]

    code_scores = {}

    for rank, idx in enumerate(top_idx):
        codes = get_law_codes_from_citations(train_gold_list[idx])
        weight = 1.0 / (rank + 1)

        for code in codes:
            code_scores[code] = code_scores.get(code, 0.0) + weight

    sorted_codes = sorted(code_scores.items(), key=lambda x: x[1], reverse=True)

    return [code for code, score in sorted_codes[:5]]

## 17. Law Corpus Retrieval

Scores all law articles against the expanded query using BM25, then boosts articles matching the detected law codes. Returns top-K candidates with scores.

In [ ]:
# =========================================================
# 17. LAW CORPUS RETRIEVAL
# =========================================================

def get_law_candidates(expanded_query, target_laws=None, top_k_docs=300):
    if target_laws is None:
        target_laws = []

    target_laws = [normalize_law_code(x) for x in target_laws]

    scores = bm25_laws.get_scores(expanded_query)

    for law_code in target_laws:
        indices = law_code_indices.get(law_code, np.array([], dtype=int))

        if len(indices) > 0:
            scores[indices] *= 1.25
            scores[indices] += 0.35

    top_k_docs = min(top_k_docs, len(scores))

    if top_k_docs <= 0:
        return [], {}

    top_idx = np.argpartition(scores, -top_k_docs)[-top_k_docs:]
    top_idx = top_idx[np.argsort(scores[top_idx])[::-1]]

    candidates = []
    score_map = {}

    for idx in top_idx:
        citation = law_citations[idx]
        score = float(scores[idx])

        if citation not in score_map:
            candidates.append(citation)
            score_map[citation] = score

    return candidates, score_map

## 17B. Train Gold Prior + Noise Filter

Builds a frequency prior from training gold citations. Also implements a strong noise filter to suppress systematically over-retrieved but irrelevant articles (e.g. BGG appeals articles for non-appeal queries).

In [ ]:
# =========================================================
# 17B. TRAIN GOLD PRIOR + STRONG NOISE FILTER
# =========================================================

train_gold_counter = Counter()

for gold_string in train[gold_col].astype(str).tolist():
    for citation in parse_citations(gold_string):
        train_gold_counter[citation] += 1

max_train_gold_count = max(train_gold_counter.values()) if len(train_gold_counter) > 0 else 1


def citation_train_prior(citation):
    count = train_gold_counter.get(citation, 0)

    if count == 0:
        return 0.0

    return math.log1p(count) / math.log1p(max_train_gold_count)


def query_is_bgg_related(query):
    q = str(query).lower()

    bgg_terms = [
        "appeal",
        "federal court",
        "federal tribunal",
        "bundesgericht",
        "admissible",
        "admissibility",
        "cassation",
        "standing to appeal",
        "time limit for appeal",
        "judicial review",
    ]

    return any(term in q for term in bgg_terms)


def query_allows_kgtg_or_vnf(query):
    q = str(query).lower()

    allowed_terms = [
        "cultural property",
        "culture",
        "museum",
        "art object",
        "archaeological",
        "heritage",
        "food",
        "nutrition",
        "feed",
        "animal",
        "agriculture",
        "veterinary",
    ]

    return any(term in q for term in allowed_terms)


def query_allows_art_754_or(query):
    q = str(query).lower()

    allowed_terms = [
        "director",
        "board",
        "shareholder",
        "company director",
        "corporate liability",
        "management liability",
        "breach of duty by directors",
        "ag",
        "corporation",
    ]

    return any(term in q for term in allowed_terms)


def is_repeated_noise_citation(citation, direct_citations, target_laws, query=""):
    citation = str(citation)

    if citation in direct_citations:
        return False

    code = extract_law_code_from_citation(citation)
    q = str(query).lower()

    noisy_exact = {
        "Art. 3 Abs. 1 221.434",
        "Art. 197e Abs. 2 AVO",
        "Art. 103 BGG",
        "Art. 105 BGG",
        "Art. 100 BGG",
        "Art. 76 BGG",
        "Art. 754 OR",
        "Art. 24 Abs. 1 KGTG",
        "Art. 43 Abs. 3 VNF",
        "Art. 43 Abs. 4 VNF",
    }

    if citation in noisy_exact:
        if code == "BGG" and query_is_bgg_related(query):
            return False

        if citation == "Art. 754 OR" and query_allows_art_754_or(query):
            return False

        if code in ["KGTG", "VNF"] and query_allows_kgtg_or_vnf(query):
            return False

        return True

    if code == "BGG" and not query_is_bgg_related(query):
        return True

    if code in ["KGTG", "VNF"] and not query_allows_kgtg_or_vnf(query):
        return True

    if citation == "Art. 754 OR" and not query_allows_art_754_or(query):
        return True

    if re.search(r"\b\d{3}\.\d+\b", citation):
        return True

    return False

## 18. Final Agentic Retrieval Function

The main pipeline: direct citation extraction → LLM query expansion → law corpus BM25 → training query transfer → merge + score + noise filter → optional LLM re-rank → top-K selection.

In [ ]:
# =========================================================
# 18. FINAL AGENTIC RETRIEVAL FUNCTION
# =========================================================

def retrieve_agentic_citations(
    query,
    query_bm25,
    train_gold_list,
    top_n_train=5,
    top_k_docs=300,
    max_citations=5,
    rerank_top_n=0
):
    query = str(query)

    direct_citations = extract_direct_citations_from_query(query)

    expanded_query, detected_laws = llm_expand_query(query)

    direct_laws = []

    for citation in direct_citations:
        code = extract_law_code_from_citation(citation)

        if code is not None:
            direct_laws.append(code)

    predicted_laws = predict_law_codes_from_similar_queries(
        query=query,
        query_bm25=query_bm25,
        train_gold_list=train_gold_list,
        top_n_train=top_n_train
    )

    target_laws = list(set(
        [normalize_law_code(x) for x in detected_laws + predicted_laws + direct_laws]
    ))

    law_candidates, law_score_map = get_law_candidates(
        expanded_query=expanded_query,
        target_laws=target_laws,
        top_k_docs=top_k_docs
    )

    train_candidates, train_support_map = retrieve_from_similar_train_queries(
        query=query,
        query_bm25=query_bm25,
        train_gold_list=train_gold_list,
        top_n_train=top_n_train,
        max_citations=30
    )

    merged = []
    seen = set()

    for citation in direct_citations + law_candidates + train_candidates:
        if citation not in seen:
            merged.append(citation)
            seen.add(citation)

    if len(merged) == 0:
        return []

    max_law_score = max(law_score_map.values()) if len(law_score_map) > 0 else 1.0

    scored = []

    for rank, citation in enumerate(merged):
        if is_repeated_noise_citation(citation, direct_citations, target_laws, query):
            continue

        law_score = law_score_map.get(citation, 0.0)
        law_score_norm = law_score / (max_law_score + 1e-9)

        train_support = train_support_map.get(citation, 0.0)
        train_prior = citation_train_prior(citation)

        code = extract_law_code_from_citation(citation)
        law_code_boost = 0.50 if code in target_laws else 0.0

        direct_boost = 8.0 if citation in direct_citations else 0.0
        rank_score = 1.0 / (rank + 1)

        final_score = (
            2.00 * law_score_norm +
            1.50 * train_support +
            1.00 * train_prior +
            0.50 * rank_score +
            law_code_boost +
            direct_boost
        )

        scored.append([citation, final_score])

    scored = sorted(scored, key=lambda x: x[1], reverse=True)

    if rerank_top_n > 0:
        rerank_pool = scored[:rerank_top_n]
        rest_pool = scored[rerank_top_n:]

        reranked = []

        for citation, base_score in rerank_pool:
            llm_score = llm_score_candidate(query, citation)
            new_score = base_score + (1.0 * llm_score)
            reranked.append([citation, new_score])

        scored = sorted(reranked + rest_pool, key=lambda x: x[1], reverse=True)

    selected = []
    seen = set()

    for citation in direct_citations:
        if citation not in seen:
            selected.append(citation)
            seen.add(citation)

        if len(selected) >= max_citations:
            return selected

    for citation, score in scored:
        if citation not in seen:
            selected.append(citation)
            seen.add(citation)

        if len(selected) >= max_citations:
            break

    return selected

## 18B. Direct Citation Test

Quick sanity check to verify that article references in queries are correctly parsed and resolved to valid citation strings.

In [ ]:
# =========================================================
# 18B. DIRECT CITATION TEST
# =========================================================

print("\nDirect citation test:")

test_q1 = "Are the claims prescribed under Art. 83 SVG and does Art. 59 Abs. 1 SVG apply?"
print(test_q1)
print(extract_direct_citations_from_query(test_q1))

test_q2 = "Does Art. 166 Abs. 2 SchKG apply?"
print(test_q2)
print(extract_direct_citations_from_query(test_q2))

test_q3 = "Can Art. 267a CO apply?"
print(test_q3)
print(extract_direct_citations_from_query(test_q3))

## 19. Hyperparameter Validation

Holds out 20% of training data, samples 30 examples, and grid-searches over `top_n_train`, `max_citations`, and `rerank_top_n` to find the best F1 configuration.

In [ ]:
# =========================================================
# 19. QUICK VALIDATION
# =========================================================

train_index_df = train
valid_df = val

train_queries_index = train_index_df[question_col].astype(str).tolist()
train_gold_index = train_index_df[gold_col].astype(str).tolist()

query_bm25_holdout = BM25(train_queries_index)

print("\nHoldout BM25 index built.")
print("Train index shape:", train_index_df.shape)
print("Validation shape:", valid_df.shape)

VALIDATION_SIZE = min(10, len(valid_df))
valid_small = valid_df.sample(VALIDATION_SIZE, random_state=42)

max_cit_values = [3, 4, 5]
top_n_values = [3, 5, 8]
rerank_values = [0]

best_f1 = -1
best_params = {
    "top_n_train": 5,
    "max_citations": 4,
    "rerank_top_n": 0
}

for top_n_train in top_n_values:
    for max_citations in max_cit_values:
        for rerank_top_n in rerank_values:
            scores_list = []

            for _, row in valid_small.iterrows():
                query = row[question_col]
                gold = parse_citations(row[gold_col])

                pred = retrieve_agentic_citations(
                    query=query,
                    query_bm25=query_bm25_holdout,
                    train_gold_list=train_gold_index,
                    top_n_train=top_n_train,
                    top_k_docs=200,
                    max_citations=max_citations,
                    rerank_top_n=rerank_top_n
                )

                score = f1_score_citations(gold, pred)
                scores_list.append(score)

            avg_f1 = float(np.mean(scores_list))

            print(
                "top_n_train =", top_n_train,
                "| max_citations =", max_citations,
                "| rerank_top_n =", rerank_top_n,
                "| F1 =", avg_f1
            )

            if avg_f1 > best_f1:
                best_f1 = avg_f1
                best_params = {
                    "top_n_train": top_n_train,
                    "max_citations": max_citations,
                    "rerank_top_n": rerank_top_n
                }

print("\nBest validation F1:", best_f1)
print("Best params:", best_params)

## 20. Final Settings

Applies the best hyperparameters found during validation for use in the test submission.

In [ ]:
# =========================================================
# 20. FINAL SETTINGS
# =========================================================

FINAL_TOP_N_TRAIN = best_params["top_n_train"]
FINAL_MAX_CITATIONS = best_params["max_citations"]
FINAL_RERANK_TOP_N = best_params["rerank_top_n"]

print("\nFinal settings:")
print("USE_LLM:", USE_LLM)
print("FINAL_TOP_N_TRAIN:", FINAL_TOP_N_TRAIN)
print("FINAL_MAX_CITATIONS:", FINAL_MAX_CITATIONS)
print("FINAL_RERANK_TOP_N:", FINAL_RERANK_TOP_N)

## 21. Create Submission

Runs the full agentic retrieval pipeline on all test queries using final settings. Saves predictions to `submission.csv`.

In [ ]:
# =========================================================
# 21. CREATE SUBMISSION
# =========================================================

test_predictions = []

for i, row in test.iterrows():
    query = row[question_col]

    pred = retrieve_agentic_citations(
        query=query,
        query_bm25=query_bm25_full,
        train_gold_list=train_gold_full,
        top_n_train=FINAL_TOP_N_TRAIN,
        top_k_docs=300,
        max_citations=FINAL_MAX_CITATIONS,
        rerank_top_n=FINAL_RERANK_TOP_N
    )

    test_predictions.append(pred)

    if (i + 1) % 10 == 0:
        print(f"Processed {i + 1}/{len(test)} test queries")


submission = pd.DataFrame({
    id_col: test[id_col],
    pred_col: [";".join(cites) for cites in test_predictions]
})

display(submission.head(10))

print("Submission shape:", submission.shape)
print("Submission columns:", submission.columns.tolist())

submission.to_csv("submission.csv", index=False)

print("\nsubmission.csv created successfully.")

## 22. Check Sample Output

Prints a few example queries and their predicted citations for a quick visual sanity check before submitting.

In [ ]:
# =========================================================
# 22. CHECK SAMPLE OUTPUT
# =========================================================

print("\nSample predictions:")

for i in range(min(5, len(test))):
    print("Query:", test.iloc[i][question_col])
    print("Prediction:", submission.iloc[i][pred_col])
    print()